
# Project 2: Female Tertiary Enrollment vs. CO₂ Per Capita

I'm exploring whether countries with higher female tertiary enrollment also tend to have higher CO₂ emissions per person, or if the two move independently. My hunch is that industrialization and income growth can raise both emissions and education access, but energy mix and policy might break the link.



## What I'm trying to answer
- For China, India, the US, and the UK, do female tertiary enrollment and CO₂ per capita rise together in overlapping years?
- Over time, who moves in tandem and who diverges?
- I want one combined visualization (animated scatter + a static snapshot) that shows both measures on the same chart.



## Data I'm using (2 provided + 1 external)
- **Dataset A (provided)** `co2-emissions-per-capita.csv`: annual CO₂ per person for China, India, UK, US, World.
- **Dataset B (provided)** `owid-co2-data.csv`: I only use this to grab ISO codes so the country names line up cleanly.
- **Dataset C (external API)** World Bank indicator `SE.TER.ENRR.FE`: female gross tertiary enrollment (%).

Why I need Dataset C: the provided files don't include education. To meet the "two datasets in one chart" requirement, I have to pull education from the World Bank, which requires internet.



## My plan
1) Load Dataset A, standardize column names, and attach ISO codes from Dataset B; drop the World aggregate.
2) Download Dataset C from the World Bank, reshape wide → long, and keep numeric years/values.
3) Merge A and C on ISO + year so I only keep overlapping observations.
4) Check Pearson correlation; build a 1990+ animated scatter with an OLS trendline.
5) Add a latest-year static scatter for easy screenshots; jot down what I see and what’s missing.


## Environment & libraries

In [1]:

import io
import zipfile
import urllib.request

import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', 10)


## Load Dataset A: CO₂ per capita

In [2]:

co2_pc = pd.read_csv('co2-emissions-per-capita.csv')
co2_pc = co2_pc.rename(columns={
    'Entity': 'country',
    'Year': 'year',
    'Annual CO₂ emissions (per capita)': 'co2_per_capita'
})
co2_pc.head()


,country,year,co2_per_capita
0,Afghanistan,1949,0.001992
1,Afghanistan,1950,0.010837
2,Afghanistan,1951,0.011625
3,Afghanistan,1952,0.011468
4,Afghanistan,1953,0.013123


## Attach ISO codes from Dataset B (clean country names)

In [3]:

iso_lookup = (
    pd.read_csv('owid-co2-data.csv', usecols=['country', 'iso_code'])
    .dropna()
    .drop_duplicates()
)
co2_pc = co2_pc.merge(iso_lookup, on='country', how='left')
# drop World aggregate and rows without ISO
co2_pc = co2_pc[co2_pc['iso_code'].notna() & (co2_pc['country'] != 'World')]
co2_pc.sample(5, random_state=0)


,country,year,co2_per_capita,iso_code
23450,Tanzania,1997,0.089440,TZA
14591,Madagascar,1963,0.083524,MDG
14778,Malaysia,1941,0.428633,MYS
2670,Belgium,1867,4.462679,BEL
26434,Zimbabwe,1950,1.139346,ZWE



## Download Dataset C: female tertiary enrollment (World Bank `SE.TER.ENRR.FE`)
I'm downloading the indicator ZIP, picking the CSV, converting wide → long, and keeping numeric years/values plus ISO codes.


In [4]:

url = 'https://api.worldbank.org/v2/en/indicator/SE.TER.ENRR.FE?downloadformat=csv'
with urllib.request.urlopen(url) as resp:
    z = zipfile.ZipFile(io.BytesIO(resp.read()))
    csv_name = [n for n in z.namelist() if n.startswith('API_SE.TER.ENRR.FE') and n.endswith('.csv')][0]
    female_raw = pd.read_csv(z.open(csv_name), skiprows=4)

female_long = female_raw.melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='year',
    value_name='female_tertiary_enrollment'
)
female_long['year'] = pd.to_numeric(female_long['year'], errors='coerce')
female_long['female_tertiary_enrollment'] = pd.to_numeric(
    female_long['female_tertiary_enrollment'], errors='coerce'
)
female_long = female_long.dropna(subset=['year', 'female_tertiary_enrollment'])
female_long = female_long.rename(columns={'Country Name': 'country_name', 'Country Code': 'iso_code'})
female_long.head()


,country_name,iso_code,year,female_tertiary_enrollment
3193,Africa Eastern and Southern,AFE,1970.0,1.60850
3194,Afghanistan,AFG,1970.0,0.22595
3195,Africa Western and Central,AFW,1970.0,0.25121
3199,Arab World,ARB,1970.0,5.75873
3201,Argentina,ARG,1970.0,11.41217



## Filter to CO₂ countries and merge (ISO + year)
I keep only the countries present in the CO₂ file, then merge on ISO and year so name differences don’t cause trouble.


In [5]:

focus_iso = co2_pc['iso_code'].unique()
female_focus = female_long[female_long['iso_code'].isin(focus_iso)]

merged = co2_pc.merge(
    female_focus,
    on=['iso_code', 'year'],
    how='inner',
    suffixes=('_co2', '_edu')
)
merged = merged[['country', 'iso_code', 'year', 'co2_per_capita', 'female_tertiary_enrollment']]

print('Countries:', merged['country'].unique())
print('Rows merged:', len(merged))
merged.head()


Countries: ['Afghanistan' 'Albania' 'Algeria' 'Andorra' 'Angola'
 'Antigua and Barbuda' 'Argentina' 'Armenia' 'Aruba' 'Australia' 'Austria'
 'Azerbaijan' 'Bahamas' 'Bahrain' 'Bangladesh' 'Barbados' 'Belarus'
 'Belgium' 'Belize' 'Benin' 'Bermuda' 'Bhutan' 'Bosnia and Herzegovina'
 'Botswana' 'Brazil' 'British Virgin Islands' 'Brunei' 'Bulgaria'
 'Burkina Faso' 'Burundi' 'Cambodia' 'Cameroon' 'Canada' 'Cape Verde'
 'Central African Republic' 'Chad' 'Chile' 'China' 'Colombia' 'Comoros'
 'Congo' 'Costa Rica' "Cote d'Ivoire" 'Croatia' 'Cuba' 'Curacao' 'Cyprus'
 'Czechia' 'Democratic Republic of Congo' 'Denmark' 'Djibouti' 'Dominica'
 'Dominican Republic' 'East Timor' 'Ecuador' 'Egypt' 'El Salvador'
 'Equatorial Guinea' 'Eritrea' 'Estonia' 'Eswatini' 'Ethiopia' 'Fiji'
 'Finland' 'France' 'French Polynesia' 'Gabon' 'Gambia' 'Georgia'
 'Germany' 'Ghana' 'Greece' 'Grenada' 'Guatemala' 'Guinea' 'Guinea-Bissau'
 'Guyana' 'Haiti' 'Honduras' 'Hong Kong' 'Hungary' 'Iceland' 'India'
 'Indonesia' 'Ira

,country,iso_code,year,co2_per_capita,female_tertiary_enrollment
0,Afghanistan,AFG,1970,0.147952,0.22595
1,Afghanistan,AFG,1972,0.129103,0.25742
2,Afghanistan,AFG,1973,0.134517,0.34517
3,Afghanistan,AFG,1974,0.153431,0.28596
4,Afghanistan,AFG,1975,0.166071,0.31291



## Quick stats
I check the overall Pearson correlation and also look at the latest year to see who’s highest on enrollment and emissions.


In [6]:

corr = merged[['female_tertiary_enrollment', 'co2_per_capita']].corr().iloc[0, 1]
print(f'Pearson correlation (all years, all countries): {corr:.2f}')

latest_year = merged['year'].max()
print(f'Latest year in merge: {latest_year}')
print(merged[merged['year'] == latest_year].sort_values('female_tertiary_enrollment', ascending=False).head())


Pearson correlation (all years, all countries): 0.35
Latest year in merge: 2024
        country iso_code  year  co2_per_capita  female_tertiary_enrollment
2348      Macao      MAC  2024        1.466572                  149.559406
1732  Hong Kong      HKG  2024        4.494174                  126.023529
40      Albania      ALB  2024        1.591990                   92.487778
2653   Mongolia      MNG  2024       12.859549                   87.794037
2624    Moldova      MDA  2024        1.755157                   84.764553



## Visualization choices
- Scatter plot to show both measures; color = country; animate by year (1990+) to see movement.
- Add an OLS trendline for direction; also make a latest-year static scatter that’s easy to drop into slides.


In [7]:

# Animated scatter (1990+)
viz_df = merged[merged['year'] >= 1990].copy()
fig = px.scatter(
    viz_df,
    x='female_tertiary_enrollment',
    y='co2_per_capita',
    color='country',
    animation_frame='year',
    hover_name='country',
    trendline='ols',
    labels={
        'female_tertiary_enrollment': 'Female tertiary enrollment (% gross)',
        'co2_per_capita': 'CO₂ per capita (tons/person)'
    },
    title='Female tertiary enrollment vs CO₂ per capita (1990+, animation)'
)
fig.update_layout(height=600)
fig.show()


In [8]:

# Latest-year static snapshot (for slides)
snapshot_year = viz_df['year'].max()
snap = viz_df[viz_df['year'] == snapshot_year]
fig2 = px.scatter(
    snap,
    x='female_tertiary_enrollment',
    y='co2_per_capita',
    color='country',
    text='country',
    labels={
        'female_tertiary_enrollment': 'Female tertiary enrollment (% gross)',
        'co2_per_capita': 'CO₂ per capita (tons/person)'
    },
    title=f'Female tertiary enrollment vs CO₂ per capita ({snapshot_year})'
)
fig2.update_traces(textposition='top center')
fig2.update_layout(height=500)
fig2.show()



## My Takeaways
Using all overlapping countries/years, the overall Pearson correlation between female tertiary enrollment and CO₂ per capita is about 0.35 (moderate positive).
Country patterns diverge: many show strong positive correlations (Algeria ~0.94, Vietnam ~0.97—education rises alongside industrialization/emissions); some are weakly positive (Bangladesh ~0.50); others are negative (Andorra ~-0.65, Zambia ~-0.75—education gains while per-capita emissions fall or stay low).

Globally there’s a modest tendency for enrollment and emissions to climb together, but the country-level spread is wide. Industrializing economies often see both rise; economies decarbonizing or already service-heavy can raise education while holding or lowering CO₂. Correlation here is descriptive—energy mix, GDP, policy, and demographics likely drive the differences.